# Composing recipes

This notebooks shows how to fill the datasets section in a [recipe](https://docs.esmvaltool.org/projects/esmvalcore/en/latest/recipe/overview.html).

In [1]:
from pathlib import Path

import yaml

from esmvalcore.config import CFG
from esmvalcore.dataset import Dataset, datasets_to_recipe
from esmvalcore.experimental.recipe import Recipe

Configure ESMValCore so it always searches the ESGF for data

In [2]:
CFG["search_data"] = "complete"

Here is a small example recipe, that uses the `datasets_to_recipe` function to convert a list of datasets to a recipe:

In [3]:
tas = Dataset(
    short_name="tas",
    mip="Amon",
    project="CMIP6",
    dataset="CanESM5-1",
    ensemble="r1i1p1f1",
    exp="historical",
    grid="gn",
    timerange="2000/2002",
)
tas["diagnostic"] = "diagnostic_name"

pr = tas.copy(short_name="pr")

print(yaml.safe_dump(datasets_to_recipe([tas, pr])))

datasets:
- dataset: CanESM5-1
diagnostics:
  diagnostic_name:
    variables:
      pr:
        ensemble: r1i1p1f1
        exp: historical
        grid: gn
        mip: Amon
        project: CMIP6
        timerange: 2000/2002
      tas:
        ensemble: r1i1p1f1
        exp: historical
        grid: gn
        mip: Amon
        project: CMIP6
        timerange: 2000/2002



A more ambitious recipe might want to use all data that is available on ESGF. We can define a dataset template with a facet value of `*` where any value can be used. This can then be expanded to a list of datasets using the `from_files()` method.

In [4]:
dataset_template = Dataset(
    short_name="tas",
    mip="Amon",
    project="CMIP6",
    exp="historical",
    dataset="*",
    institute="*",
    ensemble="*",
    grid="*",
)
datasets = list(dataset_template.from_files())
len(datasets)

1011

This results in the following recipe:

In [5]:
for dataset in datasets:
    dataset.facets["diagnostic"] = "diagnostic_name"
recipe = Path("recipe.yml")
with recipe.open("w") as file:
    yaml.safe_dump(datasets_to_recipe(datasets), file, encoding="utf-8")

print(recipe.read_text(encoding="utf-8")[:300])
print("...")

datasets:
- dataset: ACCESS-CM2
  ensemble: r(1:10)i1p1f1
  grid: gn
  institute: CSIRO-ARCCSS
- dataset: ACCESS-ESM1-5
  ensemble: r(1:40)i1p1f1
  grid: gn
  institute: CSIRO
- dataset: AWI-CM-1-1-MR
  ensemble: r(1:5)i1p1f1
  grid: gn
  institute: AWI
- dataset: AWI-ESM-1-1-LR
  ensemble: r1i1p1f1
...


The above is a rather long list of datasets, therefore we provide a function
to format recipes in a more compact way:

In [6]:
print(Recipe(recipe).to_yaml())

datasets:
  - {dataset: ACCESS-CM2, ensemble: 'r(1:10)i1p1f1', grid: gn, institute: CSIRO-ARCCSS}
  - {dataset: ACCESS-ESM1-5, ensemble: 'r(1:40)i1p1f1', grid: gn, institute: CSIRO}
  - {dataset: AWI-CM-1-1-MR, ensemble: 'r(1:5)i1p1f1', grid: gn, institute: AWI}
  - {dataset: AWI-ESM-1-1-LR, ensemble: r1i1p1f1, grid: gn, institute: AWI}
  - {dataset: AWI-ESM-1-REcoM, ensemble: r1i1p1f1, grid: gn, institute: AWI}
  - {dataset: BCC-CSM2-MR, ensemble: 'r(1:3)i1p1f1', grid: gn, institute: BCC}
  - {dataset: BCC-ESM1, ensemble: 'r(1:3)i1p1f1', grid: gn, institute: BCC}
  - {dataset: CAMS-CSM1-0, ensemble: 'r(1:2)i1p1f1', grid: gn, institute: CAMS}
  - {dataset: CAMS-CSM1-0, ensemble: r1i1p1f2, grid: gn, institute: CAMS}
  - {dataset: CAS-ESM2-0, ensemble: 'r(1:4)i1p1f1', grid: gn, institute: CAS}
  - {dataset: CESM2, ensemble: 'r(1:11)i1p1f1', grid: gn, institute: NCAR}
  - {dataset: CESM2-FV2, ensemble: 'r(1:3)i1p1f1', grid: gn, institute: NCAR}
  - {dataset: CESM2-FV2, ensemble: r1i2p2f1,